# Clasificación en Tiempo Real con Cámara Web

In [1]:
# Importo las librerías necesarias
import cv2
import numpy as np
from tensorflow import keras

2025-07-29 03:16:55.382534: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-29 03:16:55.395718: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-29 03:16:55.488926: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-29 03:16:55.608129: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753769815.731133   74613 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753769815.76

In [2]:
# Cargo el modelo entrenado
model = keras.models.load_model('91porciento.keras')
print("Modelo cargado exitosamente")
print(f"Arquitectura del modelo: {model.input_shape}")

2025-07-29 03:17:01.382025: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Modelo cargado exitosamente
Arquitectura del modelo: (None, 300, 300, 3)


In [3]:
# Defino las clases
class_names = ['bird', 'monkey', 'boar', 'tiger', 'rat', 'ram', 'dog', 'horse', 'hare', 'ox', 'dragon', 'snake']
num_classes = len(class_names)
print(f"Número de clases: {num_classes}")
print(f"Clases: {class_names}")

Número de clases: 12
Clases: ['bird', 'monkey', 'boar', 'tiger', 'rat', 'ram', 'dog', 'horse', 'hare', 'ox', 'dragon', 'snake']


In [4]:
def preprocess_frame(frame, target_size=(300, 300)):
    """
    Preprocesa el frame para la predicción
    """
    # Redimensionar la imagen
    resized = cv2.resize(frame, target_size)

    # Agregar dimensión del batch
    # Necesario por el formato de entrada del modelo
    batch_frame = np.expand_dims(resized, axis=0)

    return batch_frame

In [5]:
def draw_predictions(frame, predictions, class_names):
    """
    Dibuja las predicciones en el frame
    """
    height, width = frame.shape[:2]
    
    # Configuración del texto
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.7
    thickness = 2
    
    # Fondo para el texto
    overlay = frame.copy()
    cv2.rectangle(overlay, (10, 10), (400, 30 + len(class_names) * 30), (0, 0, 0), -1)
    frame = cv2.addWeighted(frame, 0.7, overlay, 0.3, 0)
    
    # Título
    cv2.putText(frame, "Predicciones:", (15, 30), font, font_scale, (255, 255, 255), thickness)
    
    # Mostrar cada predicción
    for i, (class_name, prob) in enumerate(zip(class_names, predictions[0])):
        text = f"{class_name}: {prob*100:.1f}%"
        y_position = 60 + i * 30
        
        # Color basado en la probabilidad
        if prob > 0.5:
            color = (0, 255, 0)  # Verde para alta probabilidad
        elif prob > 0.3:
            color = (0, 255, 255)  # Amarillo para probabilidad media
        else:
            color = (0, 0, 255)  # Rojo para baja probabilidad
        
        cv2.putText(frame, text, (15, y_position), font, font_scale, color, thickness)
    
    return frame

In [6]:
# Inicializar la cámara
cap = cv2.VideoCapture(0)

# Verificar si la cámara se abrió correctamente
if not cap.isOpened():
    print("Error: No se pudo abrir la cámara")
else:
    print("Cámara inicializada correctamente")
    print("Presiona 'q' para salir")

Cámara inicializada correctamente
Presiona 'q' para salir


In [7]:
# Loop principal de clasificación en tiempo real
try:
    i = 0
    while True:
        # Capturar frame
        ret, frame = cap.read()
        
        if not ret:
            print("Error: No se pudo capturar el frame")
            break
        
        # Preprocesar el frame
        processed_frame = preprocess_frame(frame)
        
        # Realizar predicción
        predictions = model.predict(processed_frame, verbose=0)
        
        # Dibujar predicciones en el frame
        frame_with_predictions = draw_predictions(frame, predictions, class_names)
        
        # Mostrar el frame
        cv2.imshow('Clasificacion en Tiempo Real', frame_with_predictions)
        
        # Salir con 'q'
        # Espero 20 ms para permitir la actualizacion del frame
        if cv2.waitKey(20) & 0xFF == ord('q'):
            break
    
except KeyboardInterrupt:
    print("\nInterrumpido por el usuario")

finally:
    # Limpiar recursos
    cap.release()
    cv2.destroyAllWindows()
    print("Recursos liberados")

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in "/home/don-berge/Documentos/IIA-TPS/TP-FINAL/.venv/lib/python3.12/site-packages/cv2/qt/plugins"


Recursos liberados
